In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from deel import torchlip

import pickle
import sys
sys.path.append("../..")
import liresnet.models as models
import yaml
sys.path.append("..")
from data_processing_torch import *
from notebooks_creation_models.VGG_Arthur import *

In [3]:
from auto_LiRPA import BoundedModule, BoundedTensor
from auto_LiRPA.perturbations import *

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2 moons

In [5]:
class MaxMin(nn.Module):
    """
    Custom activation layer that sorts features in pairs.
    Equivalent to torchlip.GroupSort2() but compatible with auto_LiRPA.
    """
    def forward(self, x):
        # Ensure the last dimension has an even number of features
        # if x.shape[-1] % 2 != 0:
        #     raise ValueError("The last dimension must be even for MaxMin activation.")

        # Reshape the tensor to group the last dimension into pairs
        # (batch_size, ..., num_features) -> (batch_size, ..., num_features/2, 2)
        x_pairs = x.view(*x.shape[:-1], -1, 2)

        # Separate the pairs and compute min and max
        a = x_pairs[..., 0]
        b = x_pairs[..., 1]
        min_vals = torch.min(a, b)
        max_vals = torch.max(a, b)

        # Stack them back together to form sorted pairs
        # The result has min followed by max for each pair
        sorted_pairs = torch.stack((min_vals, max_vals), dim=-1)

        # Reshape back to the original input shape
        return sorted_pairs.view(x.shape)

In [5]:
def load_model_maxmin(device):
    
    # Other Lipschitz activations are ReLU, MaxMin, GroupSort2, GroupSort.
    wass = torchlip.Sequential(
        torchlip.SpectralLinear(2, 256),
        MaxMin(),
        torchlip.SpectralLinear(256, 128),
        MaxMin(),
        torchlip.SpectralLinear(128, 64),
        MaxMin(),
        torchlip.SpectralLinear(64, 1, bias=False),
    ).to(device)

    return wass.vanilla_export()
model = load_model_maxmin(device)
model.load_state_dict(torch.load("/home/aws_install/robustess_project/lip_models/FC_2MOONS_Lip.pt", weights_only=True))
model.eval()

/home/aws_install/miniconda3/envs/lirpa_env/lib/python3.10/site-packages/deel/torchlip/modules/module.py:159: UserWarning: Sequential model contains a layer which is not a Lipschitz layer: MaxMin()
  warnings.warn(


Sequential(
  (0): Linear(in_features=2, out_features=256, bias=True)
  (1): MaxMin()
  (2): Linear(in_features=256, out_features=128, bias=True)
  (3): MaxMin()
  (4): Linear(in_features=128, out_features=64, bias=True)
  (5): MaxMin()
  (6): Linear(in_features=64, out_features=1, bias=False)
)

In [6]:
print("Loading Sample :")
   
# Define the directory and file paths
output_dir = "./../benchmark_dataset_2MOONS"
images_path = os.path.join(output_dir, "images.pkl")
targets_path = os.path.join(output_dir, "targets.pkl")

# --- Load the Tensors ---
print(f"Loading data from {output_dir}...")

# Load the images tensor
with open(images_path, 'rb') as f:
    images = pickle.load(f)

# Load the targets tensor
with open(targets_path, 'rb') as f:
    labels = pickle.load(f)

Loading Sample :
Loading data from ./../benchmark_dataset_2MOONS...


# LiResNet

In [22]:
print("loading model :")
weights = torch.load('/home/aws_install/robustess_project/lip_models/cifar10-12x512_799.pth').get('backbone')
with open('/home/aws_install/robustess_project/liresnet/configs/cifar10.yaml', 'r') as f:
        cfg = yaml.load(f, Loader=yaml.Loader)
model_cfg = cfg['model']
dataset_cfg = cfg['dataset']
gloro_cfg = cfg['gloro']
model = models.GloroNet(**model_cfg, **dataset_cfg).to(device)
model.load_state_dict(weights)
model.eval()

loading model :


GloroNet(
  (stem): Sequential(
    (0): Conv2d(3, 512, kernel_size=(5, 5), stride=(2, 2), padding=(2, 2), output_padding=(1, 1))
    (1): MinMax(dim=1)
  )
  (conv): LiResConv(
    depth=12, width=512, centering=True
    (act): MinMax(dim=1)
  )
  (neck): Map2Vec(
    (activation): MinMax(dim=1)
  )
  (linear): LiResMLP(
    depth=8, width=2048
    (act): MinMax(dim=1)
  )
  (head): head(in_features=2048, out_features=10, bias=True)
)

In [23]:
print("Loading Sample :")
   
# Define the directory and file paths
output_dir = "./../benchmark_dataset"
images_path = os.path.join(output_dir, "images.pkl")
targets_path = os.path.join(output_dir, "targets.pkl")

# --- Load the Tensors ---
print(f"Loading data from {output_dir}...")

# Load the images tensor
with open(images_path, 'rb') as f:
    images = pickle.load(f)

# Load the targets tensor
with open(targets_path, 'rb') as f:
    labels = pickle.load(f)

Loading Sample :
Loading data from ./../benchmark_dataset...


# VGG

In [6]:
from torch.nn.modules.utils import _pair
from typing import Optional, Union
from torch.nn.common_types import _size_2_t

# --- Helper Module for a LiRPA-compatible export ---
# This module encapsulates the LiRPA-compatible operations so that the exported
# model does not depend on our custom ScaledL2NormPool2d class definition.
class _ExportedL2Pool(nn.Module):
    def __init__(self, kernel_size, stride, ceil_mode, coeff):
        super().__init__()
        self.kernel_size = kernel_size
        self.stride = stride
        self.ceil_mode = ceil_mode
        self.coeff = coeff
        self.num_elements = self.kernel_size[0] * self.kernel_size[1]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_squared = torch.pow(x, 2)
        avg_of_squares = F.avg_pool2d(
            x_squared,
            kernel_size=self.kernel_size,
            stride=self.stride,
            ceil_mode=self.ceil_mode
        )
        sum_of_squares = avg_of_squares * self.num_elements
        pooled = torch.sqrt(sum_of_squares + 1e-9)
        return pooled * self.coeff

    def __repr__(self):
        return (f"_ExportedL2Pool(kernel_size={self.kernel_size}, "
                f"stride={self.stride}, coeff={self.coeff})")
    
def computePoolScalingFactor(kernel_size):
    if isinstance(kernel_size, tuple):
        scalingFactor = math.sqrt(np.prod(np.asarray(kernel_size)))
    else:
        scalingFactor = kernel_size
    return scalingFactor

class ScaledL2NormPool2d(torch.nn.Module, torchlip.module.LipschitzModule):
    def __init__(
        self,
        kernel_size: _size_2_t,
        stride: Optional[_size_2_t] = None,
        ceil_mode: bool = False,
        k_coef_lip: float = 1.0,
    ):
        """
        auto_LiRPA-compatible L2-norm pooling layer.
        """
        # We no longer inherit from LPPool2d, but directly from our custom base class
        # and nn.Module (via LipschitzModule).
        torch.nn.Module.__init__(self)
        torchlip.module.LipschitzModule.__init__(self, k_coef_lip)
        
        self.kernel_size = _pair(kernel_size)
        self.stride = _pair(stride) if stride is not None else self.kernel_size
        self.ceil_mode = ceil_mode

        self.scalingFactor = computePoolScalingFactor(self.kernel_size)

        if self.stride != self.kernel_size:
            raise RuntimeError("For provable robustness, stride must be equal to kernel_size for this implementation.")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 1. Square the input tensor element-wise.
        # This is a basic operation that auto_LiRPA can handle.
        x_squared = torch.pow(x, 2)
        
        # 2. Apply average pooling.
        # auto_LiRPA has native support for AvgPool2d.
        sum_squared = F.avg_pool2d(
            x_squared,
            kernel_size=self.kernel_size,
            stride=self.stride,
            ceil_mode=self.ceil_mode
        )
        
        # 3. Get the number of elements in the pooling window.
        num_elements_in_kernel = self.kernel_size[0] * self.kernel_size[1]
        
        # avg_pool(x^2) = (sum(x^2)) / N  =>  sum(x^2) = avg_pool(x^2) * N
        sum_squared = sum_squared * num_elements_in_kernel
        
        # 4. Take the element-wise square root.
        # torch.sqrt is also a standard supported operation.
        # Adding a small epsilon for numerical stability to avoid sqrt(0) gradients issues.
        pooled = torch.sqrt(sum_squared + 1e-8)
        
        # 5. Apply the Lipschitz scaling factor.
        return pooled * self._coefficient_lip * self.scalingFactor
        
    def __repr__(self):
        return (f"ScaledL2NormPool2d(kernel_size={self.kernel_size}, "
                f"stride={self.stride}, k_coef_lip={self._coefficient_lip})")

    # def vanilla_export(self):
    #     # # The vanilla export should now also return a standard LPPool2d
    #     # # for compatibility if needed elsewhere.
    #     # if self._coefficient_lip == 1.0:
    #     #     return torch.nn.LPPool2d(
    #     #         2,  # Norm 2
    #     #         kernel_size=self.kernel_size,
    #     #         stride=self.stride,
    #     #         ceil_mode=self.ceil_mode,
    #     #     )
    #     # else:
    #     #     # If the coefficient is not 1, a direct export to a single standard layer
    #     #     # is not possible. Returning self is a reasonable fallback.
    #         return self
    
    def vanilla_export(self) -> nn.Module:
        """
        Exports the layer to a self-contained, auto_LiRPA-compatible nn.Module.

        This function returns a new module that encapsulates the exact same
        LiRPA-compatible operations as this layer's forward pass. This is
        somewhat redundant, as this layer itself is already compatible.
        The primary use for this would be to create a model with no custom
        class definitions before saving or deployment.

        IMPORTANT: For LiRPA analysis, you can use the main ScaledL2NormPool2d
        layer directly. You do not need to call this export function first.
        """
        # This returns a new, standard nn.Module that is also LiRPA-compatible.
        return _ExportedL2Pool(
            kernel_size=self.kernel_size,
            stride=self.stride,
            ceil_mode=self.ceil_mode,
            coeff=self._coefficient_lip
        )


In [94]:
def load_model_deprecated():
    coeff_total = 500 # Change according to temperature
    nb_blocks = 12
    coeff_block = coeff_total**(1/nb_blocks)
    width = 1.5  # 1: 2.5M parameters, 2: 10M parameters, 4: 40M parameters

    model = torchlip.Sequential(
        LipBlock(3, int(width*64), activation=MaxMin(), coeff=coeff_block),
        LipBlock(int(width*64), int(width*64), activation=MaxMin(), coeff=coeff_block),
        LipBlock(int(width*64), int(width*64), activation=torch.abs, coeff=coeff_block),
        torchlip.ScaledL2NormPool2d(kernel_size=2, stride=2),
        LipBlock(int(width*64), int(width*128), activation=MaxMin(), coeff=coeff_block),
        LipBlock(int(width*128), int(width*128), activation=MaxMin(), coeff=coeff_block),
        LipBlock(int(width*128), int(width*128), activation=torch.abs, coeff=coeff_block),
        torchlip.ScaledL2NormPool2d(kernel_size=2, stride=2),
        LipBlock(int(width*128), int(width*256), activation=MaxMin(), coeff=coeff_block),
        LipBlock(int(width*256), int(width*256), activation=MaxMin(), coeff=coeff_block),
        LipBlock(int(width*256), int(width*256), activation=torch.abs, coeff=coeff_block),
        torchlip.ScaledL2NormPool2d(kernel_size=2, stride=2),
        LipBlock(int(width*256), int(width*512), activation=MaxMin(), coeff=coeff_block),
        LipBlock(int(width*512), int(width*512), activation=MaxMin(), coeff=coeff_block),
        LipBlock(int(width*512), int(width*512), activation=torch.abs, coeff=coeff_block),
        torchlip.ScaledAdaptativeL2NormPool2d((1, 1)),
        nn.Flatten(),
        torchlip.SpectralLinear(int(width*512), 10),
        MultiplyByScalar(1/coeff_total)
    )
    return model

In [7]:
class _ExportedAdaptiveL2Pool(nn.Module):
    def __init__(self, output_size, coeff):
        super().__init__()
        self.output_size = output_size
        self.coeff = coeff
        self.adaptive_avg_pool = nn.AdaptiveAvgPool2d(output_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Get spatial dimensions to calculate total number of elements
        h, w = x.shape[-2:]
        num_elements = h * w

        # LiRPA-compatible L2 norm calculation
        x_squared = torch.pow(x, 2)
        # adaptive_avg_pool computes sum(x^2) / num_elements
        avg_of_squares = self.adaptive_avg_pool(x_squared)
        sum_of_squares = avg_of_squares * num_elements
        pooled = torch.sqrt(sum_of_squares + 1e-9)
        
        return pooled * self.coeff

    def __repr__(self):
        return (f"_ExportedAdaptiveL2Pool(output_size={self.output_size}, "
                f"coeff={self.coeff})")
    
class ScaledAdaptiveL2NormPool2d(torch.nn.Module, torchlip.module.LipschitzModule):
    def __init__(
        self,
        output_size: _size_2_t = (1, 1),
        k_coef_lip: float = 1.0,
    ):
        """
        auto_LiRPA-compatible Adaptive L2-norm pooling layer.

        This layer's forward pass is implemented using only operations natively
        supported by auto_LiRPA (pow, adaptive_avg_pool2d, sqrt, mul).
        """
        torch.nn.Module.__init__(self)
        torchlip.module.LipschitzModule.__init__(self, k_coef_lip)
        
        # Ensure output_size is a tuple of two integers
        if not isinstance(output_size, tuple) or len(output_size) != 2:
             output_size = _pair(output_size)

        # For this operation to be a valid norm, it must collapse the spatial dimensions
        if output_size[0] != 1 or output_size[1] != 1:
            raise ValueError("output_size must be (1, 1) for ScaledAdaptiveL2NormPool2d")
        
        self.output_size = output_size
        # We use the standard AdaptiveAvgPool2d as a supported building block
        self.adaptive_avg_pool = nn.AdaptiveAvgPool2d(self.output_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Performs adaptive L2 pooling using a sequence of LiRPA-compatible operations.
        """
        # 1. Get spatial dimensions to calculate total number of elements.
        h, w = x.shape[-2:]
        num_elements = h * w

        # 2. Square the input tensor.
        x_squared = torch.pow(x, 2)
        
        # 3. Apply adaptive average pooling to the squared tensor.
        # This computes (sum of squares) / num_elements
        avg_of_squares = self.adaptive_avg_pool(x_squared)
        
        # 4. Multiply by num_elements to get the sum of squares.
        sum_of_squares = avg_of_squares * num_elements
        
        # 5. Take the square root to get the L2 norm over the spatial dimensions.
        # Add a small epsilon for numerical stability.
        pooled = torch.sqrt(sum_of_squares + 1e-9)
        
        # 6. Apply the Lipschitz scaling factor.
        return pooled * self._coefficient_lip
        
    def __repr__(self):
        return (f"ScaledAdaptiveL2NormPool2d(output_size={self.output_size}, "
                f"k_coef_lip={self._coefficient_lip})")

    def vanilla_export(self) -> nn.Module:
        """
        Exports the layer to a self-contained, auto_LiRPA-compatible nn.Module.
        """
        return _ExportedAdaptiveL2Pool(
            output_size=self.output_size,
            coeff=self._coefficient_lip
        )

In [8]:
def load_model():
    coeff_total = 500 # Change according to temperature
    nb_blocks = 12
    coeff_block = coeff_total**(1/nb_blocks)
    width = 1.5  # 1: 2.5M parameters, 2: 10M parameters, 4: 40M parameters

    model = torchlip.Sequential(
        LipBlock(3, int(width*64), activation=MaxMin(), coeff=coeff_block),
        LipBlock(int(width*64), int(width*64), activation=MaxMin(), coeff=coeff_block),
        LipBlock(int(width*64), int(width*64), activation=torch.abs, coeff=coeff_block),
        ScaledL2NormPool2d(kernel_size=2, stride=2),
        LipBlock(int(width*64), int(width*128), activation=MaxMin(), coeff=coeff_block),
        LipBlock(int(width*128), int(width*128), activation=MaxMin(), coeff=coeff_block),
        LipBlock(int(width*128), int(width*128), activation=torch.abs, coeff=coeff_block),
        ScaledL2NormPool2d(kernel_size=2, stride=2),
        LipBlock(int(width*128), int(width*256), activation=MaxMin(), coeff=coeff_block),
        LipBlock(int(width*256), int(width*256), activation=MaxMin(), coeff=coeff_block),
        LipBlock(int(width*256), int(width*256), activation=torch.abs, coeff=coeff_block),
        ScaledL2NormPool2d(kernel_size=2, stride=2),
        LipBlock(int(width*256), int(width*512), activation=MaxMin(), coeff=coeff_block),
        LipBlock(int(width*512), int(width*512), activation=MaxMin(), coeff=coeff_block),
        LipBlock(int(width*512), int(width*512), activation=torch.abs, coeff=coeff_block),
        ScaledAdaptiveL2NormPool2d((1, 1)),
        nn.Flatten(),
        torchlip.SpectralLinear(int(width*512), 10),
        MultiplyByScalar(1/coeff_total)
    )
    return model

In [9]:
print("loading model :")
model = load_model().vanilla_export().to(device)

model.load_state_dict(torch.load('/home/aws_install/robustess_project/lip_notebooks/notebooks_creation_models/Vgg_lip_multisteplr_van.pt', weights_only=True))
model.eval()

loading model :


/home/aws_install/miniconda3/envs/lirpa_env/lib/python3.10/site-packages/deel/torchlip/modules/module.py:159: UserWarning: Sequential model contains a layer which is not a Lipschitz layer: LipBlock(
  (conv): ParametrizedSpectralConv2d(
    3, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect
    (parametrizations): ModuleDict(
      (weight): ParametrizationList(
        (0): _SpectralNorm()
        (1): _BjorckNorm()
        (2): _LConvNorm()
      )
    )
  )
  (norm): BatchCentering()
  (activation): MaxMin()
  (scalar): MultiplyByScalar()
)
  warnings.warn(
/home/aws_install/miniconda3/envs/lirpa_env/lib/python3.10/site-packages/deel/torchlip/modules/module.py:159: UserWarning: Sequential model contains a layer which is not a Lipschitz layer: LipBlock(
  (conv): ParametrizedSpectralConv2d(
    96, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect
    (parametrizations): ModuleDict(
      (weight): Parametr

Sequential(
  (0): LipBlock(
    (conv): ParametrizedSpectralConv2d(
      3, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect
      (parametrizations): ModuleDict(
        (weight): ParametrizationList(
          (0): _SpectralNorm()
          (1): _BjorckNorm()
          (2): _LConvNorm()
        )
      )
    )
    (norm): BatchCentering()
    (activation): MaxMin()
    (scalar): MultiplyByScalar()
  )
  (1): LipBlock(
    (conv): ParametrizedSpectralConv2d(
      96, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect
      (parametrizations): ModuleDict(
        (weight): ParametrizationList(
          (0): _SpectralNorm()
          (1): _BjorckNorm()
          (2): _LConvNorm()
        )
      )
    )
    (norm): BatchCentering()
    (activation): MaxMin()
    (scalar): MultiplyByScalar()
  )
  (2): LipBlock(
    (conv): ParametrizedSpectralConv2d(
      96, 96, kernel_size=(3, 3), stride=(1, 1), padding=(

In [11]:
from torchsummary import summary
summary(model, (3,32,32))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
     _SpectralNorm-1              [-1, 3, 3, 3]               0
       _BjorckNorm-2              [-1, 3, 3, 3]               0
        _LConvNorm-3              [-1, 3, 3, 3]               0
ParametrizedSpectralConv2d-4           [-1, 96, 32, 32]           2,592
     _SpectralNorm-5              [-1, 3, 3, 3]               0
       _BjorckNorm-6              [-1, 3, 3, 3]               0
        _LConvNorm-7              [-1, 3, 3, 3]               0
     _SpectralNorm-8              [-1, 3, 3, 3]               0
       _BjorckNorm-9              [-1, 3, 3, 3]               0
       _LConvNorm-10              [-1, 3, 3, 3]               0
    _SpectralNorm-11              [-1, 3, 3, 3]               0
      _BjorckNorm-12              [-1, 3, 3, 3]               0
       _LConvNorm-13              [-1, 3, 3, 3]               0
    _SpectralNorm-14           

In [95]:
print("loading model :")
model_dep = load_model_deprecated().vanilla_export().to(device)

model_dep.load_state_dict(torch.load('/home/aws_install/robustess_project/lip_notebooks/notebooks_creation_models/Vgg_lip_multisteplr_van.pt', weights_only=True))
model_dep.eval()

loading model :


/home/aws_install/miniconda3/envs/lirpa_env/lib/python3.10/site-packages/deel/torchlip/modules/module.py:159: UserWarning: Sequential model contains a layer which is not a Lipschitz layer: LipBlock(
  (conv): ParametrizedSpectralConv2d(
    3, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect
    (parametrizations): ModuleDict(
      (weight): ParametrizationList(
        (0): _SpectralNorm()
        (1): _BjorckNorm()
        (2): _LConvNorm()
      )
    )
  )
  (norm): BatchCentering()
  (activation): MaxMin()
  (scalar): MultiplyByScalar()
)
  warnings.warn(
/home/aws_install/miniconda3/envs/lirpa_env/lib/python3.10/site-packages/deel/torchlip/modules/module.py:159: UserWarning: Sequential model contains a layer which is not a Lipschitz layer: LipBlock(
  (conv): ParametrizedSpectralConv2d(
    96, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect
    (parametrizations): ModuleDict(
      (weight): Parametr

Sequential(
  (0): LipBlock(
    (conv): ParametrizedSpectralConv2d(
      3, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect
      (parametrizations): ModuleDict(
        (weight): ParametrizationList(
          (0): _SpectralNorm()
          (1): _BjorckNorm()
          (2): _LConvNorm()
        )
      )
    )
    (norm): BatchCentering()
    (activation): MaxMin()
    (scalar): MultiplyByScalar()
  )
  (1): LipBlock(
    (conv): ParametrizedSpectralConv2d(
      96, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect
      (parametrizations): ModuleDict(
        (weight): ParametrizationList(
          (0): _SpectralNorm()
          (1): _BjorckNorm()
          (2): _LConvNorm()
        )
      )
    )
    (norm): BatchCentering()
    (activation): MaxMin()
    (scalar): MultiplyByScalar()
  )
  (2): LipBlock(
    (conv): ParametrizedSpectralConv2d(
      96, 96, kernel_size=(3, 3), stride=(1, 1), padding=(

In [67]:
print("Loading Sample :")
   
# Define the directory and file paths
output_dir = "./../benchmark_dataset"
images_path = os.path.join(output_dir, "images.pkl")
targets_path = os.path.join(output_dir, "targets.pkl")

# --- Load the Tensors ---
print(f"Loading data from {output_dir}...")

# Load the images tensor
with open(images_path, 'rb') as f:
    images = pickle.load(f)

# Load the targets tensor
with open(targets_path, 'rb') as f:
    labels = pickle.load(f)

Loading Sample :
Loading data from ./../benchmark_dataset...


ModuleNotFoundError: No module named 'deel.lip'

# Generation of Lirpa bounds

In [93]:
model(images[100:101].to(device))

tensor([[-0.0093, -0.0214, -0.0154, -0.0223, -0.0295, -0.0313, -0.0135, -0.0309,
         -0.0279, -0.0369]], device='cuda:0')

In [96]:
model_dep(images[100:101].to(device))

tensor([[-0.0093, -0.0214, -0.0154, -0.0223, -0.0295, -0.0313, -0.0135, -0.0309,
         -0.0279, -0.0369]], device='cuda:0', grad_fn=<MulBackward0>)

In [97]:
bounded_model = BoundedModule(model, torch.ones_like(images[100:101]).to(device))
bounded_model.eval()

/home/aws_install/miniconda3/envs/lirpa_env/lib/python3.10/site-packages/orthogonium/layers/normalization.py:81: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if self.running_num_batches == 0:
/home/aws_install/miniconda3/envs/lirpa_env/lib/python3.10/site-packages/torch/onnx/_internal/jit_utils.py:307: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not applied. (Triggered internally at ../torch/csrc/jit/passes/onnx/constant_fold.cpp:179.)
  _C._jit_pass_onnx_node_shape_type_inference(node, params_dict, opset_version)
/home/aws_install/miniconda3/envs/lirpa_env/lib/python3.10/site-packages/torch/onnx/utils.py:702: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onn

In [98]:
bounded_model.device

device(type='cuda', index=0)

In [100]:
eps = 0.06
norm = 2
ptb = PerturbationLpNorm(norm = norm, eps = eps)
# Input tensor is wrapped in a BoundedTensor object.
bounded_image = BoundedTensor(images[100:101], ptb).to(device)

In [103]:
print('Bounding method: backward (CROWN, DeepPoly)')
with torch.no_grad():  # If gradients of the bounds are not needed, we can use no_grad to save memory.
  lb, ub = bounded_model.compute_bounds(x=(bounded_image,), method='CROWN-IBP')
print(lb, ub)

Bounding method: backward (CROWN, DeepPoly)


IndexError: tuple index out of range